# Player Stat Distribution Explorer

This notebook uses `players_2022-23_to_2025-26.csv` to explore a player's game-by-game stat distribution.

Use the widgets to choose:
- a player
- a season
- a target stat

The output includes:
- a game-by-game plot for the selected season
- a histogram / PMF view of the selected stat
- a Q-Q plot against a fitted normal distribution
- a simple distribution-fit table so you can see whether the sample looks more normal, Poisson-like, or negative-binomial-like

In [1]:
from pathlib import Path
import os
import warnings

project_root = Path.cwd()
mpl_dir = project_root / ".matplotlib"
mpl_dir.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_dir.resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

plt.style.use("ggplot")
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [2]:
DATASET_NAME = "players_2022-23_to_2025-26.csv"


def find_dataset(filename=DATASET_NAME):
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / "box_scores" / "all_seasons" / "2022-23_to_2025-26" / filename,
        Path.cwd() / "my_proj" / "data" / "box_scores" / "all_seasons" / "2022-23_to_2025-26" / filename,
        Path.cwd().parent / filename,
    ]
    for path in candidates:
        if path.exists():
            return path.resolve()

    matches = list(Path.cwd().rglob(filename))
    if matches:
        return matches[0].resolve()

    raise FileNotFoundError(f"Could not find {filename} from {Path.cwd()}")


def minutes_to_float(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)
    value = str(value).strip()
    if ":" not in value:
        try:
            return float(value)
        except ValueError:
            return np.nan
    minutes, seconds = value.split(":", maxsplit=1)
    try:
        return int(minutes) + int(seconds) / 60.0
    except ValueError:
        return np.nan


players_path = find_dataset()
players = pd.read_csv(players_path)
players["gameDate"] = pd.to_datetime(players["gameDate"], errors="coerce")
players["minutesFloat"] = players["minutes"].map(minutes_to_float)
players["playerName"] = (
    players["firstName"].fillna("").str.strip() + " " + players["familyName"].fillna("").str.strip()
).str.strip()
players.loc[players["playerName"].eq(""), "playerName"] = players.loc[players["playerName"].eq(""), "playerSlug"].fillna("Unknown")
players = players.sort_values(["playerName", "season", "gameDate", "gameId"]).reset_index(drop=True)

exclude_numeric = {"gameId", "teamId", "personId"}
numeric_stat_cols = []
for col in players.columns:
    if col in exclude_numeric:
        continue
    numeric_version = pd.to_numeric(players[col], errors="coerce")
    if numeric_version.notna().sum() > 0:
        numeric_stat_cols.append(col)

preferred_stats = [
    "points",
    "assists",
    "reboundsTotal",
    "threePointersMade",
    "fieldGoalsMade",
    "freeThrowsMade",
    "minutesFloat",
    "steals",
    "blocks",
    "turnovers",
    "plusMinusPoints",
    "usagePercentage",
    "trueShootingPercentage",
]

ordered_stats = [col for col in preferred_stats if col in numeric_stat_cols] + [
    col for col in numeric_stat_cols if col not in preferred_stats
]

print(f"Loaded {len(players):,} rows from: {players_path}")
print(f"Found {len(players['playerName'].unique()):,} players and {len(ordered_stats)} numeric stat columns.")

Loaded 128,621 rows from: /Users/devlincorrigan/nyu/8sp26/ML/group project/my_proj/data/box_scores/all_seasons/2022-23_to_2025-26/players_2022-23_to_2025-26.csv
Found 892 players and 44 numeric stat columns.


In [3]:
def is_nonnegative_integer_series(values, tol=1e-9):
    arr = np.asarray(values, dtype=float)
    if len(arr) == 0:
        return False
    if np.any(arr < -tol):
        return False
    return np.all(np.abs(arr - np.round(arr)) < tol)


def fit_distributions(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    results = []
    if len(x) < 3:
        return pd.DataFrame(columns=["distribution", "params", "log_likelihood", "aic"])

    def add_result(name, params, loglik, k):
        if np.isfinite(loglik):
            results.append(
                {
                    "distribution": name,
                    "params": params,
                    "log_likelihood": float(loglik),
                    "aic": float(2 * k - 2 * loglik),
                }
            )

    mu, sigma = stats.norm.fit(x)
    sigma = max(float(sigma), 1e-8)
    add_result("Normal", f"mu={mu:.2f}, sigma={sigma:.2f}", np.sum(stats.norm.logpdf(x, loc=mu, scale=sigma)), 2)

    if np.all(x > 0):
        try:
            gamma_a, gamma_loc, gamma_scale = stats.gamma.fit(x, floc=0)
            add_result(
                "Gamma",
                f"shape={gamma_a:.2f}, scale={gamma_scale:.2f}",
                np.sum(stats.gamma.logpdf(x, gamma_a, loc=gamma_loc, scale=gamma_scale)),
                2,
            )
        except Exception:
            pass

        try:
            lognorm_s, lognorm_loc, lognorm_scale = stats.lognorm.fit(x, floc=0)
            add_result(
                "Lognormal",
                f"shape={lognorm_s:.2f}, scale={lognorm_scale:.2f}",
                np.sum(stats.lognorm.logpdf(x, lognorm_s, loc=lognorm_loc, scale=lognorm_scale)),
                2,
            )
        except Exception:
            pass

    if is_nonnegative_integer_series(x):
        xr = np.round(x).astype(int)
        lam = max(float(xr.mean()), 1e-8)
        add_result("Poisson", f"lambda={lam:.2f}", np.sum(stats.poisson.logpmf(xr, mu=lam)), 1)

        mean_x = float(xr.mean())
        var_x = float(xr.var(ddof=1)) if len(xr) > 1 else 0.0
        if var_x > mean_x and mean_x > 0:
            n = mean_x ** 2 / (var_x - mean_x)
            p = n / (n + mean_x)
            if n > 0 and 0 < p < 1:
                add_result(
                    "Negative Binomial",
                    f"n={n:.2f}, p={p:.3f}",
                    np.sum(stats.nbinom.logpmf(xr, n, p)),
                    2,
                )

    return pd.DataFrame(results).sort_values("aic", kind="stable").reset_index(drop=True)


def distribution_summary(values):
    x = pd.Series(values, dtype=float).dropna()
    if x.empty:
        return "No data available for this selection."

    mean_x = x.mean()
    var_x = x.var(ddof=1) if len(x) > 1 else 0.0
    skew_x = x.skew() if len(x) > 2 else np.nan
    integer_like = is_nonnegative_integer_series(x)

    notes = [
        f"games={len(x)}",
        f"mean={mean_x:.2f}",
        f"std={x.std(ddof=1):.2f}" if len(x) > 1 else "std=NA",
        f"skew={skew_x:.2f}" if pd.notna(skew_x) else "skew=NA",
    ]

    if integer_like:
        if np.isfinite(var_x) and mean_x > 0:
            ratio = var_x / mean_x
            notes.append(f"variance/mean={ratio:.2f}")
            if abs(ratio - 1) < 0.2:
                notes.append("shape hint: close to Poisson dispersion")
            elif ratio > 1.2:
                notes.append("shape hint: overdispersed count data, often negative-binomial-like")
            else:
                notes.append("shape hint: underdispersed relative to Poisson")
    else:
        if abs(skew_x) < 0.5:
            notes.append("shape hint: fairly symmetric")
        elif skew_x > 0:
            notes.append("shape hint: right-skewed")
        else:
            notes.append("shape hint: left-skewed")

    return " | ".join(notes)

In [4]:
common_players = sorted(players["playerName"].dropna().unique())
common_seasons = sorted(players["season"].dropna().astype(str).unique())

player_dropdown = widgets.Combobox(
    options=common_players,
    value="James Harden" if "James Harden" in common_players else common_players[0],
    description="Player",
    ensure_option=True,
    layout=widgets.Layout(width="320px"),
)
season_dropdown = widgets.Dropdown(
    options=common_seasons,
    value="2025-26" if "2025-26" in common_seasons else common_seasons[-1],
    description="Season",
    layout=widgets.Layout(width="180px"),
)
stat_dropdown = widgets.Dropdown(
    options=ordered_stats,
    value="points" if "points" in ordered_stats else ordered_stats[0],
    description="Target stat",
    layout=widgets.Layout(width="260px"),
)
bins_slider = widgets.IntSlider(
    value=15,
    min=5,
    max=40,
    step=1,
    description="Bins",
    continuous_update=False,
    layout=widgets.Layout(width="250px"),
)
line_input = widgets.FloatText(
    value=25.0,
    description="O/U line",
    step=0.5,
    layout=widgets.Layout(width="180px"),
)
output = widgets.Output()


def update_season_options(*_):
    player_df = players.loc[players["playerName"] == player_dropdown.value]
    seasons = sorted(player_df["season"].dropna().astype(str).unique())
    if seasons:
        season_dropdown.options = seasons
        if season_dropdown.value not in seasons:
            season_dropdown.value = seasons[-1]


player_dropdown.observe(update_season_options, names="value")
update_season_options()

In [5]:
def render(player_name, season, stat_col, bins, line_value):
    with output:
        clear_output(wait=True)

        df = players.loc[
            (players["playerName"] == player_name) &
            (players["season"].astype(str) == str(season))
        ].copy()

        if df.empty:
            display(Markdown("No games found for that player-season selection."))
            return

        df[stat_col] = pd.to_numeric(df[stat_col], errors="coerce")
        df = df.dropna(subset=[stat_col]).sort_values(["gameDate", "gameId"])

        if df.empty:
            display(Markdown(f"`{stat_col}` has no numeric values for this player-season."))
            return

        values = df[stat_col].to_numpy(dtype=float)
        fit_table = fit_distributions(values)
        summary = distribution_summary(values)
        integer_like = is_nonnegative_integer_series(values)
        under_prob = float(np.mean(values < line_value))
        push_prob = float(np.mean(values == line_value))
        over_prob = float(np.mean(values > line_value))

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        ax_ts, ax_hist, ax_qq, ax_box = axes.ravel()

        game_numbers = np.arange(1, len(df) + 1)
        ax_ts.plot(game_numbers, values, marker="o", linewidth=2, markersize=5, color="#1f77b4")
        ax_ts.axhline(values.mean(), linestyle="--", linewidth=1.5, color="#d62728", label=f"Mean = {values.mean():.2f}")
        ax_ts.axhline(line_value, linestyle=":", linewidth=1.5, color="#2ca02c", label=f"O/U line = {line_value:.2f}")
        ax_ts.set_title(f"{player_name} - {stat_col} by game ({season})")
        ax_ts.set_xlabel("Game number")
        ax_ts.set_ylabel(stat_col)
        ax_ts.legend(loc="best")

        if integer_like:
            ints = np.round(values).astype(int)
            min_x, max_x = ints.min(), ints.max()
            grid = np.arange(min_x, max_x + 1)
            counts = pd.Series(ints).value_counts(normalize=True).sort_index()
            ax_hist.bar(counts.index, counts.values, width=0.8, alpha=0.45, color="#4c78a8", label="Empirical PMF")

            if not fit_table.empty:
                for name in fit_table["distribution"].head(3):
                    if name == "Poisson":
                        lam = ints.mean()
                        ax_hist.plot(grid, stats.poisson.pmf(grid, mu=lam), marker="o", linewidth=2, label=f"{name} fit")
                    elif name == "Negative Binomial":
                        mean_x = float(ints.mean())
                        var_x = float(ints.var(ddof=1)) if len(ints) > 1 else 0.0
                        if var_x > mean_x and mean_x > 0:
                            n = mean_x ** 2 / (var_x - mean_x)
                            p = n / (n + mean_x)
                            ax_hist.plot(grid, stats.nbinom.pmf(grid, n, p), marker="o", linewidth=2, label=f"{name} fit")
                    elif name == "Normal":
                        mu, sigma = stats.norm.fit(values)
                        sigma = max(sigma, 1e-8)
                        xgrid = np.linspace(min_x - 1, max_x + 1, 400)
                        ax_hist.plot(xgrid, stats.norm.pdf(xgrid, mu, sigma), linewidth=2, label=f"{name} fit")

            ax_hist.set_xlabel(stat_col)
            ax_hist.set_ylabel("Probability")
        else:
            ax_hist.hist(values, bins=bins, density=True, alpha=0.45, color="#4c78a8", edgecolor="white", label="Histogram")
            xgrid = np.linspace(values.min(), values.max(), 400)
            for name in fit_table["distribution"].head(3):
                if name == "Normal":
                    mu, sigma = stats.norm.fit(values)
                    sigma = max(sigma, 1e-8)
                    ax_hist.plot(xgrid, stats.norm.pdf(xgrid, mu, sigma), linewidth=2, label=f"{name} fit")
                elif name == "Gamma" and np.all(values > 0):
                    a, loc, scale = stats.gamma.fit(values, floc=0)
                    ax_hist.plot(xgrid, stats.gamma.pdf(xgrid, a, loc=loc, scale=scale), linewidth=2, label=f"{name} fit")
                elif name == "Lognormal" and np.all(values > 0):
                    s, loc, scale = stats.lognorm.fit(values, floc=0)
                    ax_hist.plot(xgrid, stats.lognorm.pdf(xgrid, s, loc=loc, scale=scale), linewidth=2, label=f"{name} fit")

            ax_hist.set_xlabel(stat_col)
            ax_hist.set_ylabel("Density")

        ax_hist.axvline(line_value, linestyle=":", linewidth=2, color="#2ca02c", label=f"O/U line = {line_value:.2f}")
        ax_hist.set_title(f"Distribution of {stat_col}")
        ax_hist.legend(loc="best")

        (osm, osr), (slope, intercept, r) = stats.probplot(values, dist="norm")
        ax_qq.scatter(osm, osr, s=28, alpha=0.8)
        ax_qq.plot(osm, slope * np.asarray(osm) + intercept, color="#d62728", linewidth=2)
        ax_qq.set_title(f"Normal Q-Q plot (r = {r:.3f})")
        ax_qq.set_xlabel("Theoretical quantiles")
        ax_qq.set_ylabel("Ordered values")

        ax_box.boxplot(values, vert=True, patch_artist=True, boxprops={"facecolor": "#72b7b2", "alpha": 0.6})
        ax_box.set_title(f"Boxplot of {stat_col}")
        ax_box.set_ylabel(stat_col)
        ax_box.set_xticks([])

        fig.suptitle(f"{player_name} | {season} | {stat_col}", fontsize=16, y=1.02)
        fig.tight_layout()
        plt.show()

        display(Markdown(f"**Summary:** {summary}"))
        display(Markdown(
            f"**Empirical O/U at {line_value:.2f}:** "
            f"Under = {under_prob:.1%} | Push = {push_prob:.1%} | Over = {over_prob:.1%}"
        ))
        display(df[["gameDate", "teamTricode", stat_col]].rename(columns={"teamTricode": "team"}).reset_index(drop=True).tail(10))

        if fit_table.empty:
            display(Markdown("Not enough data to compare fitted distributions."))
        else:
            display(Markdown("**Candidate distribution fits** (lower AIC is better within this table):"))
            display(fit_table)

In [6]:
top_row = widgets.HBox([player_dropdown, season_dropdown, stat_dropdown])
bottom_row = widgets.HBox([bins_slider, line_input])
controls = widgets.VBox([top_row, bottom_row])

def refresh(*_):
    render(player_dropdown.value, season_dropdown.value, stat_dropdown.value, bins_slider.value, line_input.value)

for widget in [player_dropdown, season_dropdown, stat_dropdown, bins_slider, line_input]:
    widget.observe(refresh, names="value")

display(controls)
display(output)
refresh()

Output()

## Example

Try:
- `Player = James Harden`
- `Season = 2025-26`
- `Target stat = points`

That will show Harden's points in each game that season and help you judge whether the distribution looks closer to a count distribution like Poisson / negative binomial or a smoother bell-shaped distribution.